# Bài 26 — chuỗi luyện Easy · LeetCode #125 — Valid Palindrome

**Mức độ:** Easy · **Ngôn ngữ:** Python 3 · **Đề gốc:** https://leetcode.com/problems/valid-palindrome/

**Mục tiêu học tập**

- Đọc đề và tự phân biệt input/output, constraints và ví dụ biên.
- Nắm cú pháp Python theo từng cell thực nghiệm trước khi ghép thuật toán.
- Hiểu brute force/baseline trước, sau đó giải thích mấu chốt tối ưu bằng bất biến.
- Trace từng bước với dữ liệu dễ sai, không xem code như một hộp đen.
- Chạy lời giải chuẩn LeetCode, test mẫu, edge case, randomized/exhaustive khi phù hợp.
- Tự giải **6 bài luyện từ dễ đến khó**, chỉ xem gợi ý/đáp án cuối sau khi thử.

**Cách học:** đọc Markdown → tự dự đoán output → chạy cell Python ngay sau đó → giải thích lại bằng lời. Các cell TODO không chạy tự động và chỉ phục vụ tự luyện.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho chuỗi `s`, bỏ qua mọi ký tự **không phải chữ hoặc chữ số**, so sánh **không phân biệt chữ hoa/thường**; trả True nếu chuỗi còn lại là đối xứng.

### Ví dụ quan trọng

`"A man, a plan, a canal: Panama" → True`; `"race a car" → False`; `" " → True` vì sau khi bỏ qua ký tự thì chuỗi rỗng.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import List, Optional
from collections import deque, Counter

### 2.1. Chuỗi `str`, vị trí ký tự và `len`

`str` là một chuỗi ký tự theo thứ tự, đánh chỉ số từ 0.
Với `s="ab3!"`, `s[0]` là `'a'`, `s[3]` là `'!'`, `len(s)==4`.
Chỉ số cuối luôn bằng `len(s)-1`, không phải `len(s)`.
`s[-1]` cũng truy cập ký tự cuối, nhưng không được dùng khi s rỗng.
Python `str` **immutable** (không sửa riêng từng ký tự); muốn tạo chuỗi đã lọc phải tạo object mới.
Trước khi làm bài, hãy tập viết `s[left]` và `s[right]` với hai đầu chuỗi.

In [ ]:
example_string='ab3!'
print('độ dài:',len(example_string),'đầu:',example_string[0],'cuối:',example_string[-1])
print('các cặp chỉ số:',list(enumerate(example_string)))
assert example_string[len(example_string)-1]=='!'

### 2.2. `isalnum`, `isalpha`, `isdigit` — giống và khác nhau

`c.isalnum()` trả `True` nếu c là **chữ hoặc chữ số**. `isalpha()` chỉ nhận chữ,
còn `isdigit()` chỉ nhận chữ số; vì đề yêu cầu cả hai nên **không thể dùng mỗi `isalpha`**.
Với ký tự một chữ số như `'7'`, `isalnum()` là True, `isalpha()` là False.
Dấu phẩy, dấu cách, dấu hai chấm đều là False ở `isalnum()`.
Trong phạm vi đề gốc, đầu vào là các ký tự ASCII có thể in; Python cũng có
`isalnum()` cho Unicode, nhưng phần Unicode mở rộng **không phải ràng buộc của bài gốc**.
Quan sát input/output rõ ràng thay vì học thuộc tên hàm.

In [ ]:
for ch in ['A','b','9','0',' ',':','!']:
    print(repr(ch),'isalnum=',ch.isalnum(),'isalpha=',ch.isalpha(),'isdigit=',ch.isdigit())
assert '0'.isalnum() and not '0'.isalpha() and not ','.isalnum()

### 2.3. Chữ hoa/chữ thường: `lower()` không sửa chuỗi gốc

`'A'.lower()` là `'a'`, còn `'a'.lower()` vẫn là `'a'`.
Khi so hai ký tự chữ ở hai đầu, chỉ cần so sánh kết quả sau khi `.lower()`;
không cần chuyển toàn bộ chuỗi thành bản chữ thường trước.
`lower()` trả về một string mới (có thể trùng object ở một số trường hợp tối ưu,
nhưng đừng dựa vào nhận định về identity). Đặt tên rõ biến `left_char`, `right_char`.
Lưu ý chữ số qua `.lower()` không thay đổi, đúng yêu cầu bài toán.

In [ ]:
original='A2z'
changed=original.lower()
print('gốc:',original,'sau lower:',changed,'gốc vẫn:',original)
print('so sánh:', 'A'.lower()=='a'.lower(), '7'.lower()=='7')
assert changed=='a2z' and original=='A2z'

### 2.4. `join`, generator và tạo chuỗi đã lọc

`''.join(...)` ghép một dãy ký tự thành một string duy nhất.
Biểu thức `ch.lower() for ch in s if ch.isalnum()` là một **generator expression**:
đọc từng ký tự, chỉ giữ ký tự hợp lệ, rồi đổi chữ thường.
Cách này rất trực giác, phù hợp làm baseline để kiểm chứng thuật toán tối ưu.
Tuy vậy, chuỗi `clean` phải lưu thêm `O(n)` ký tự trong trường hợp xấu nhất.
Vì đầu ra chỉ là True/False, ta có động cơ tìm cách không tạo chuỗi sạch.

In [ ]:
raw_text='A man, a plan!'
chars=[ch.lower() for ch in raw_text if ch.isalnum()]
cleaned=''.join(chars)
print('list ký tự:',chars)
print('string đã lọc:',repr(cleaned),'độ dài:',len(cleaned))
assert cleaned=='amanaplan'

### 2.5. `s[::-1]` và chi phí của slicing

`s[start:stop:step]` tạo một **lát cắt**; với `step=-1` và bỏ start/stop,
`s[::-1]` trả chuỗi đảo chiều. Nó không đảo tại chỗ vì string không thể sửa.
Ví dụ `'abc'[::-1] == 'cba'`. Cách so `clean == clean[::-1]` kiểm tra đối xứng đúng,
nhưng thêm một chuỗi đảo nữa nên dùng `O(n)` bộ nhớ.
Khi đề yêu cầu tối ưu bộ nhớ, hai con trỏ tốt hơn vì chỉ giữ 2 chỉ số.
Hai cách vẫn có cùng bậc thời gian `O(n)`, khác nhau về extra space.

In [ ]:
for part in ('abc','aba','', '12321'):
    print(repr(part),'->',repr(part[::-1]),'đối xứng?',part==part[::-1])
assert 'abc'[::-1]=='cba' and ''[::-1]==''

### 2.6. `while` và hai con trỏ hội tụ

**Two Pointers** là hai vị trí `left` và `right` cùng tham gia một phép kiểm tra.
Bắt đầu `left=0`, `right=len(s)-1`; mỗi vòng chọn một trong ba thao tác:
bỏ ký tự trái không hợp lệ, bỏ ký tự phải không hợp lệ, hoặc so một cặp hợp lệ.
Khi so xong một cặp bằng nhau, cập nhật `left += 1`, `right -= 1`.
Điều kiện `while left < right` đúng cả chuỗi rỗng / một ký tự: vòng lặp không chạy
vì không còn cặp đối diện cần kiểm tra. **Bất biến** là vùng ngoài hai con trỏ đã ổn.

In [ ]:
left,right=0,6
while left<right:
    print('cặp chỉ số:',left,right)
    left+=1;right-=1
print('kết thúc:',left,right)
assert left>=right

### 2.7. `continue`, `return` và hai điều kiện không được đảo

`continue` bỏ phần còn lại của **vòng hiện tại** và quay lại kiểm tra `while`.
Khi gặp dấu phẩy ở bên trái, ta chỉ tăng `left` rồi `continue`;
không được ngay lập tức đọc `s[right]` như thể đã có ký tự trái hợp lệ.
`return False` thì kết thúc cả hàm ngay khi thấy **hai chữ/số khác nhau**.
`return True` chỉ được đặt **sau vòng while**, nghĩa là không còn đôi đối xứng nào sai.
Thực hành điều khiển luồng trước khi đọc một thuật toán dài sẽ tránh những lỗi khó phát hiện.

In [ ]:
for value in [1,2,3,4,5]:
    if value==2:
        print('bỏ qua',value)
        continue
    if value==4:
        print('dừng sớm tại',value)
        break
    print('xử lý',value)
print('khác biệt: break ra khỏi vòng; return ra khỏi hàm')

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

`char.isalnum()` kiểm tra chữ/số; `char.lower()` đổi sang chữ thường; `s[::-1]` đảo chuỗi nhưng tạo chuỗi mới. Hai con trỏ `left,right` đi dần vào giữa.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
for c in ['A','7',':',' ']:print(repr(c),'isalnum?',c.isalnum(),'lower:',c.lower())
s='A b!3';clean=''.join(c.lower() for c in s if c.isalnum())
print('sau lọc:',repr(clean),'đảo:',clean[::-1])

### Câu hỏi gợi mở trước khi tối ưu

Sau khi tự chạy ví dụ nền tảng, hãy tự trả lời bằng lời:

1. Trong bài **Valid Palindrome**, đâu là dữ liệu thật, đâu là công cụ dựng thử?
2. Biến nào thay đổi sau một vòng lặp / sau một lời gọi đệ quy?
3. Khi nào một ký tự / node / giá trị được xem là 'đã xử lý xong'?
4. Nếu input rỗng, có một phần tử, hay chỉ có một nhánh, cần return kiểu gì?
5. Thao tác nào tăng theo số phần tử n, thao tác nào chỉ giữ vài biến?

Không cần trả lời bằng công thức ngay; thử input nhỏ rồi diễn đạt kết quả.

### Quy trình chạy notebook không phụ thuộc trạng thái ngầm

Các hàm dựng input, baseline, `solve`, trace và đáp án được định nghĩa theo thứ tự từ trên xuống dưới.
Không chạy phần đáp án trước phần bài tập nếu muốn tự luyện.
Khi thử một biến thể, hãy tạo input mới, không sửa input đã được dùng cho `assert` ở các cell phía trên.
Nếu chạy lại một cell, bạn nên kiểm tra các biến demo bị ghi đè; cách ổn định nhất là **Restart Kernel & Run All**.

Ở các bài Linked List/Tree, LeetCode đã cung cấp sẵn lớp node. Trong notebook, lớp node được định nghĩa để mô phỏng local và không phải mã cần nộp.

### 3.1. Tự lọc ký tự bằng vòng `for` trước khi học generator

Trước khi dùng `''.join(ch.lower() for ch in s if ch.isalnum())`,
hãy thử **viết rõ mỗi thao tác**: tạo list trống, duyệt từng ký tự,
dùng `if` loại dấu câu, rồi `.append()` chữ đã chuẩn hóa.
Bài toán palindrome cần loại bỏ dấu câu vì câu tiếng Anh thường có khoảng trắng, dấu phẩy.
Cách viết tường minh dài hơn nhưng tốt để debug: có thể in `ch` và hành động tương ứng.
Đây cũng là mô hình để suy ra giải pháp một dòng sau khi đã hiểu bản chất.

In [ ]:
raw='Ab, 12!'
pieces=[]
for ch in raw:
    if ch.isalnum():
        pieces.append(ch.lower())
        print(repr(ch),'-> giữ')
    else:
        print(repr(ch),'-> bỏ')
print('kết quả:', ''.join(pieces))
assert ''.join(pieces)=='ab12'

### 3.2. Quy tắc xác nhận một cặp đối diện

Sau khi đã bỏ qua ký tự không hợp lệ ở hai đầu,
hai con trỏ phải dừng ở **ký tự chữ hoặc số**.
Nếu một cặp khác nhau sau `.lower()`, toàn bộ chuỗi không đối xứng,
nên có thể `return False` ngay mà không phải kiểm tra phần còn lại.
Nếu cặp bằng nhau, ta loại cặp ấy khỏi vùng cần xem và đưa hai con trỏ vào giữa.
Sự tiến triển bắt buộc: mỗi vòng ít nhất một con trỏ đổi vị trí, nên không lặp vô hạn.
Dự đoán cặp đầu tiên của `'A!a'`, `'0P'`, `'abc!CBA'` trước khi chạy.

In [ ]:
examples=['A!a','0P','abc!CBA']
for s0 in examples:
    i,j=0,len(s0)-1
    while i<j and not s0[i].isalnum():i+=1
    while i<j and not s0[j].isalnum():j-=1
    print(repr(s0),'->',(repr(s0[i]),repr(s0[j])) if i<j else 'không có cặp')

## 4. Cách nghĩ trực giác (baseline)

Tạo chuỗi sạch `clean`, rồi so sánh `clean == clean[::-1]`. Dễ đọc, nhưng phải tạo chuỗi phụ kích thước O(n).

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(s):
    clean=''.join(c.lower() for c in s if c.isalnum())
    return clean==clean[::-1]

In [ ]:
for x in ['A man, a plan, a canal: Panama','race a car',' ']:print(repr(x),'->',baseline(x))

### 4.1. Brute Force: tách chuỗi đã lọc rồi đảo

Cách trực giác xử lý thành hai giai đoạn độc lập:
(1) lọc bỏ dấu câu, chuẩn hóa chữ; (2) so chuỗi với bản đảo.
Mỗi bước dễ hiểu, dễ chạy thử, dễ viết kiểm thử độc lập.
Nhưng lọc ký tự `O(n)` tạo chuỗi độ dài tối đa n,
đảo cũng `O(n)` và sinh chuỗi tạm mới; bộ nhớ phụ `O(n)`.
Đừng vội tối ưu nếu chưa tự giải thích vì sao công thức `'aba' == 'aba'[::-1]` đúng.
`''` cũng là palindrome theo quy tắc đối xứng rỗng.

In [ ]:
for phrase in ('racecar','Race, car!','A man, a plan, a canal: Panama','  '):
    normalized=''.join(c.lower() for c in phrase if c.isalnum())
    print(repr(phrase),'->',repr(normalized),'đảo:',repr(normalized[::-1]),'KQ:',baseline(phrase))
assert baseline('  ')==True

### 4.2. Phản ví dụ: `s == s[::-1]` trên chuỗi gốc có thể sai

Nếu bỏ qua bước lọc, `'A man, a plan, a canal: Panama'` cho kết quả False,
dù theo định nghĩa bài toán nó phải là True.
Lý do: ký tự hoa và dấu cách còn giữ nguyên, nên so sánh string không tương đương yêu cầu.
Hãy đối chiếu **bốn cách**: so raw, so lowercase raw, so filtered, và solve hai con trỏ.
Chỉ cách sau cùng vừa đúng vừa không cần cấp phát chuỗi phụ tỉ lệ n.
Phân biệt `đúng về logic` và `tối ưu tài nguyên` trong từng bước suy luận.

In [ ]:
phrase='A man, a plan, a canal: Panama'
print('raw đảo:',phrase==phrase[::-1])
print('chỉ lower:',phrase.lower()==phrase.lower()[::-1])
cleaned=''.join(c.lower() for c in phrase if c.isalnum())
print('lọc + lower:',cleaned==cleaned[::-1])
print('baseline:',baseline(phrase))
assert cleaned==cleaned[::-1]

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Không cần tạo chuỗi mới: dùng **hai con trỏ từ hai đầu**. Nếu gặp ký tự không phải chữ/số thì bỏ qua; nếu là chữ/số thì so sánh sau `lower()` rồi cùng tiến.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

### Bước đột phá — Giữ nguyên chuỗi, chỉ di chuyển hai vị trí

Ở baseline, chúng ta **xây dựng một chuỗi sạch** chỉ để so hai đầu.
Nhưng khi so một chuỗi đối xứng, mỗi lần chỉ cần nhìn **hai ký tự** tại thời điểm đó.
Nếu không hợp lệ, một con trỏ dịch vào; nếu hợp lệ, so sánh sau chuẩn hóa.
Do đó có thể giữ nguyên `s`, lưu đúng hai chỉ số `left` và `right` thay vì tạo bản sao.

**Bất biến trước mỗi vòng:** mọi cặp ký tự hợp lệ nằm *ngoài đoạn* `[left, right]`
đã được đối chiếu và khớp; các dấu câu bỏ qua không ảnh hưởng bài toán.
Nếu phát hiện cặp hợp lệ không khớp, đáp án chắc chắn False.
Nếu `[left, right]` rỗng hoặc chỉ còn một ký tự, không còn cặp nào có thể sai.

## 6. Lời giải chính — hiểu từng bước rồi mới viết

`left=0,right=len(s)-1`. Khi gặp ký tự không hợp lệ ở phía nào thì dịch riêng con trỏ đó; nếu cả hai hợp lệ thì so sánh. Mismatch trả False; hai con trỏ gặp/cắt nhau trả True.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Mô hình trạng thái: đầu trái, đầu phải, hành động

Mỗi vòng chỉ cần biết `left`, `right`, ký tự tại hai vị trí và điều kiện hợp lệ.
Nếu `s[left].isalnum()` False, dịch **left**; không thay right.
Ngược lại, nếu `s[right].isalnum()` False, dịch **right**.
Chỉ khi **cả hai hợp lệ** mới so sau `lower()` và dịch **cả hai**.
Ta không được bỏ qua dấu câu ở hai đầu rồi mặc định so ngay khi chưa kiểm tra lại.
Thử lập bảng thao tác với `'A!,a'`: từ đầu sẽ chọn nhánh nào?

In [ ]:
example='A!,a';left,right=0,len(example)-1
decisions=[]
while left<right:
    if not example[left].isalnum(): action='skip-left';left+=1
    elif not example[right].isalnum(): action='skip-right';right-=1
    else:
        action='compare '+repr(example[left])+','+repr(example[right])
        left+=1;right-=1
    decisions.append(action)
print('quyết định:',decisions)

In [ ]:
def solve(s: str) -> bool:
    left,right=0,len(s)-1
    while left<right:
        if not s[left].isalnum():left+=1;continue
        if not s[right].isalnum():right-=1;continue
        if s[left].lower()!=s[right].lower():return False
        left+=1;right-=1
    return True

### 6.1. Chứng minh thuật toán luôn tiến tới điều kiện dừng

Tại mỗi lượt `while`, có đúng một trong ba trường hợp:
(a) left tăng 1; (b) right giảm 1; (c) cả left và right tiến vào giữa.
Do đó độ dài vùng chưa xử lý giảm tối thiểu 1.
Ban đầu có n vị trí, nên số lượt không quá n (không có vòng lặp vô hạn).
Đây đồng thời là lập luận **Time O(n)**, không phải `O(n²)` dù có hai con trỏ.
Phân biệt số vòng lặp với số con trỏ: hai con trỏ không có nghĩa hai vòng lặp lồng nhau.

In [ ]:
for s0 in ('A!!a','!!!','abCba','0P'):
    left,right=0,len(s0)-1
    total_steps=0
    while left<right:
        old_gap=right-left
        if not s0[left].isalnum():left+=1
        elif not s0[right].isalnum():right-=1
        else:left+=1;right-=1
        assert right-left<old_gap
        total_steps+=1
    print(repr(s0),'số lượt tối đa:',total_steps,'<= n=',len(s0))

### 6.2. Vì sao có thể dừng ngay tại cặp đầu tiên khác?

Một chuỗi đối xứng **bắt buộc** mọi cặp vị trí đối diện phải có cùng chữ/số
sau khi loại bỏ các ký tự không liên quan. Do đó một cặp sai đủ bác bỏ toàn bộ.
Việc tìm thấy một cặp đúng chưa đủ kết luận True: cần tiếp tục tới khi hết cặp.
Chúng ta có thể lưu vị trí cặp đầu tiên không khớp để debug (đó cũng là một biến thể).
Phần giải chuẩn không cần lưu vị trí, chỉ cần trả boolean.
Ví dụ `'abca'` cặp a/a đúng nhưng b/c sai ở vòng sau.

In [ ]:
for phrase in ('abca','abba','ab!ca'):
    print(repr(phrase),'baseline:',baseline(phrase),'two pointers:',solve(phrase))
    assert baseline(phrase)==solve(phrase)

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| left | right | ký tự | hành động |
|---:|---:|---|---|
| 0 | 4 | `A` / `a` | bằng nhau sau lower |
| 1 | 3 | `!` / `b` | bỏ qua `!`, left tăng |
| 2 | 3 | `b` / `b` | bằng nhau |
| 3 | 2 | giao nhau | dừng True |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [ ]:
def trace_pal(s):
    i,j=0,len(s)-1
    while i<j:
        print(f'i={i}, j={j}, chars={s[i]!r},{s[j]!r}')
        if not s[i].isalnum():i+=1;continue
        if not s[j].isalnum():j-=1;continue
        if s[i].lower()!=s[j].lower():print('KHÁC!');return False
        i+=1;j-=1
    print('KHÔNG CÓ ĐÔI NÀO KHÁC');return True
trace_pal('A!b:a')

### 7.1. Trace dạng bảng: hai đầu được kiểm tra TRƯỚC khi cập nhật

Bảng trace giúp nhìn thấy cả hai nhánh bỏ qua dấu và nhánh đối chiếu.
Mỗi dòng gồm `left trước`, `right trước`, `ch trái`, `ch phải`, `hành động`, `trạng thái sau`.
Ở input `'A!b:a'`:
- Bước 1: A/a khớp → dịch hai phía;
- Bước 2: `!` không hợp lệ → dịch trái;
- Bước 3: `:` không hợp lệ → dịch phải;
- Khi chỉ còn một chữ b, điều kiện `left < right` sai.
Hãy chạy hàm tạo bảng tự động và đối chiếu với suy luận trên.

In [ ]:
def trace_table125(s):
    i,j=0,len(s)-1; rows=[]
    while i<j:
        before=(i,j,repr(s[i]),repr(s[j]))
        if not s[i].isalnum(): act='bỏ trái';i+=1
        elif not s[j].isalnum(): act='bỏ phải';j-=1
        elif s[i].lower()!=s[j].lower():
            rows.append((*before,'KHÁC!',i,j));break
        else:act='khớp';i+=1;j-=1
        rows.append((*before,act,i,j))
    for row in rows:print(row)
    return rows
trace_table125('A!b:a')

### 7.2. Trace phản ví dụ `0P`: chữ và chữ số đều hợp lệ

Ký tự `'0'` là số, không được bỏ qua. Ký tự `'P'` là chữ, sau `lower()` thành `'p'`.
Vì `'0' != 'p'`, hàm phải trả False ngay.
Nếu dùng `isalpha()` để lọc thay cho `isalnum()`, chuỗi `'0P'` bị biến thành `'p'`
và kết luận True: đây là **lỗi sai thật**, không chỉ khác hiệu năng.
So sánh output baseline đúng, solve đúng, với bản lọc sai để nhìn phản ví dụ nhỏ nhất.
Luôn chọn input biên có chữ số khi đề nói alphanumeric.

In [ ]:
candidate='0P'
wrong_clean=''.join(c.lower() for c in candidate if c.isalpha())
print('lọc sai isalpha:',repr(wrong_clean),'kết luận:',wrong_clean==wrong_clean[::-1])
print('lọc đúng:',baseline(candidate),'hai con trỏ:',solve(candidate))
assert not solve(candidate) and (wrong_clean==wrong_clean[::-1])

### 7.3. Trường hợp chuỗi sau lọc rỗng hoặc một ký tự

Nếu input chỉ chứa dấu câu, xóa hết sẽ cho `''`: đảo chuỗi rỗng vẫn là `''`,
nên đáp án **True** theo định nghĩa của bài.
Nếu chuỗi sau lọc chỉ còn một chữ, nó cũng luôn đối xứng.
Với hai con trỏ, thuật toán tự xử lý được vì liên tục bỏ dấu câu cho tới khi các con trỏ giao nhau.
Không cần tạo một nhánh riêng cho dấu câu ở đầu hoặc cuối.
Chúng ta kiểm cả những trường hợp mở rộng có `''` dù đề gốc quy định chuỗi không rỗng.

In [ ]:
for case in ('',' ','!@#$','A','@@Z...','::0::'):
    print(repr(case),'->',solve(case),'cleaned=',repr(''.join(c.lower() for c in case if c.isalnum())))
    assert solve(case)

### 7.4. So sánh hai cách giải theo từng bước, không chỉ đáp án

Hai cách có cách nhìn khác nhau: baseline đi qua *mọi ký tự* để tạo string,
còn two pointers chỉ đọc những ký tự cần kiểm tra và dừng sớm khi phát hiện sai.
`baseline(s)` dễ dùng làm **oracle**: nếu solve không trùng trên một input nhỏ,
ta in input và bảng trace để xác định nơi suy luận bị lệch.
Với input có lỗi ngay ở hai đầu, two pointers có thể kết thúc rất nhanh,
dù worst-case vẫn là O(n). Một kết quả chạy nhanh không tự chứng minh đúng,
vì test chỉ chứng minh trên các ví dụ đã chọn.

In [ ]:
tests=['x___y','RaceCar','A man, a plan, a canal: Panama','a!b','0p','...a...']
for s0 in tests:
    x,y=baseline(s0),solve(s0)
    print(repr(s0),'baseline=',x,'two-pointers=',y)
    assert x==y

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Bất biến: mọi cặp ký tự chữ/số đã đi qua ở hai đầu đều tương ứng và bằng nhau sau chuẩn hóa. Ký tự bị bỏ qua không ảnh hưởng đến chuỗi xét. Nếu gặp cặp khác nhau, chuỗi không đối xứng; nếu hai con trỏ gặp nhau mà không gặp sai, mọi cặp đã khớp.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
class Solution:
    def isPalindrome(self, s: str) -> bool:
        left, right = 0, len(s) - 1
        while left < right:
            if not s[left].isalnum():
                left += 1
                continue
            if not s[right].isalnum():
                right -= 1
                continue
            if s[left].lower() != s[right].lower():
                return False
            left += 1
            right -= 1
        return True

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
cases=[('A man, a plan, a canal: Panama',True),('race a car',False),(' ',True),('0P',False),('a',True),('a!a',True),('',True)]
for s,expected in cases:
    assert solve(s)==expected and Solution().isPalindrome(s)==expected
    assert baseline(s)==expected
print('PASS 7 ca palindrome')

### 10.1. Kiểm thử vét cạn tập chuỗi ngắn (*exhaustive test*)

Một vài ví dụ nổi tiếng không đảm bảo bạn đã kiểm mọi nhánh.
Ta chọn alphabet nhỏ gồm chữ thường, chữ hoa, số, dấu câu: `'aA1!'`.
Dùng `itertools.product` liệt kê toàn bộ chuỗi độ dài 0 đến 6.
Với mỗi chuỗi, kết quả `solve` phải bằng `baseline`.
Việc giới hạn alphabet và độ dài khiến bộ test chạy nhanh mà vẫn chạm các nhánh:
bỏ trái, bỏ phải, so bằng nhau, so khác nhau, rỗng, số và chữ.
Đây là phương pháp học cách test logic, không phải thay thế chứng minh.

In [ ]:
from itertools import product
alphabet125='aA1!'
checked=0
for n in range(7):
    for chars in product(alphabet125,repeat=n):
        s0=''.join(chars)
        assert solve(s0)==baseline(s0),(s0,solve(s0),baseline(s0))
        checked+=1
print('PASS exhaustive',checked,'chuỗi ngắn')

### 10.2. Kiểm thử thuộc tính: thêm dấu câu không đổi đáp án

Nếu dấu câu sẽ bị bỏ hoàn toàn, việc thêm `'!!!'` vào bất kỳ vị trí nào
không làm thay đổi dãy ký tự chữ/số sau lọc, nên không được đổi kết quả.
Ta có thể dùng **metamorphic testing**: biến đổi input theo một quy tắc
bảo toàn đáp án rồi kiểm lại trên hai input liên quan.
Một biến đổi nữa là đổi chữ sang hoa: `.lower()` khiến kết quả bất biến.
Lưu ý chỉ dùng dấu không phải alphanumeric; thêm chữ hoặc số mới có thể đổi đáp án.

In [ ]:
import random
rng125=random.Random(125)
for _ in range(400):
    base=''.join(rng125.choice('ab01') for _ in range(rng125.randrange(0,12)))
    k=rng125.randrange(len(base)+1)
    decorated=base[:k]+'!?,'+base[k:]
    assert solve(base)==solve(decorated)
    assert solve(base)==solve(base.upper())
print('PASS 400 kiểm thử bảo toàn kết quả')

### 10.3. Không sửa input và các ca cực dài

String của Python immutable nên `solve` không thể thay `s` tại từng vị trí.
Tuy nhiên ta vẫn nên xác nhận hàm không dựa vào trạng thái cũ trong notebook.
Gọi `solve` nhiều lần trên input khác nhau phải cho kết quả như nhau.
Độ dài 20.000 trong kiểm thử thử tải vẫn chạy theo tuyến tính,
tránh xuất từng ký tự làm nghẽn notebook. Các test lớn chỉ in **tóm tắt**.
Thời gian đo chỉ tham khảo vì phụ thuộc máy; chứng minh Big-O mới là phân tích chính.

In [ ]:
long_pal='a'*10000+'b'+'a'*10000
long_not=long_pal[:-1]+'c'
assert solve(long_pal) is True
assert solve(long_not) is False
for candidate in ('a!b:a','0P','!!!'):
    v1=solve(candidate);v2=solve(candidate)
    assert v1==v2
print('PASS ca dài và tính ổn định qua nhiều lần gọi')

## 11. Time / Space Complexity — phân biệt bộ nhớ của baseline và tối ưu

Gọi `n = len(s)`. **Baseline:** lọc `s` thành chuỗi sạch và đảo, Time **O(n)**, extra Space **O(n)** vì cấp phát dữ liệu phụ tỉ lệ n.
**Two pointers:** mỗi lượt tiến ít nhất một con trỏ; tổng lượt và tổng số ký tự đọc đều `O(n)`; Time **O(n)**, extra Space **O(1)** cho `left,right` và ký tự tạm. Chuỗi input không bị sửa.
Nếu gặp cặp khác ngay ở hai đầu, có thể return sớm, nhưng **worst case** vẫn O(n).
Không khẳng định thời gian O(log n) vì hai con trỏ tiến từ hai phía; tổng quãng đường vẫn có thể lên tới n.

## 12. Các lỗi người mới hay mắc

- ❌ Không bỏ qua dấu cách, dấu phẩy.
- ❌ Chỉ gọi `lower()` mà không loại bỏ dấu câu.
- ❌ Dùng `isalnum()` sai thành `isalpha()` (bỏ cả chữ số).
- ❌ Quên trả True cho chuỗi toàn dấu cách.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

In [ ]:
sample_bug125='0P'
print('đầu vào dễ sai:',repr(sample_bug125),'two pointers:',solve(sample_bug125),
      'baseline:',baseline(sample_bug125))
assert solve(sample_bug125)==baseline(sample_bug125)==False

### 12.1. Bẫy loại bỏ dấu câu nhưng quên `lower`

Một hàm sai thường lọc đúng ký tự nhưng **không chuyển về cùng kiểu chữ**:
`'Aa'` sẽ thành `'Aa'` rồi so với `'aA'`, cho False.
Thay vì thêm một ngoại lệ riêng cho `'Aa'`, sửa quy tắc tổng quát:
chuẩn hóa **mọi** ký tự hợp lệ trước khi so.
Để tìm lỗi nhanh, dùng phản ví dụ nhỏ nhất với hai chữ khác case.
Hãy chạy hàm sai dưới đây để quan sát đầu ra cụ thể.

In [ ]:
def wrong_case_sensitive(s):
    clean=''.join(c for c in s if c.isalnum())
    return clean==clean[::-1]
print('wrong("Aa"):',wrong_case_sensitive('Aa'),'correct:',solve('Aa'))
assert wrong_case_sensitive('Aa') is False and solve('Aa') is True

### 12.2. Khi nào `isalpha` và `isalnum` cho kết quả trái nhau?

Người mới dễ hiểu nhầm 'chữ và chữ số' thành 'chữ'.
Dùng `isalpha()` sẽ loại các chữ số như `'0'`, `'7'`, và tạo ra chuỗi sai.
Phản ví dụ `'0P'` rất hữu ích vì kết luận bị đổi ngay.
Trên hệ thống test chính thức, đầu vào là ASCII, và `.isalnum()` phù hợp.
Đừng cố làm giải pháp nặng hơn bằng các bộ lọc ký tự phức tạp khi đề chưa yêu cầu.
Mẹo: mỗi lỗi cần **một input cụ thể** và giải thích vì sao bản sửa sửa được cả nhóm lỗi.

In [ ]:
def wrong_alpha_only(s):
    clean=''.join(ch.lower() for ch in s if ch.isalpha())
    return clean==clean[::-1]
for sample in ('0P','1a1','7!8'):
    print(repr(sample),'sai:',wrong_alpha_only(sample),'đúng:',solve(sample))
assert wrong_alpha_only('0P') != solve('0P')

## 13. Nhận diện dạng bài và vận dụng

Từ khóa **palindrome**, so sánh hai đầu và cần O(1) bộ nhớ → two pointers. Có thể bỏ qua ký tự không cần xét bằng `continue`.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tự viết hàm trả chỉ số cặp chữ/số đầu tiên không khớp thay vì chỉ trả True/False, áp dụng hai con trỏ và bỏ dấu câu.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
for s in ['a!a','1a2','Aa']:
    print(repr(s),solve(s))

### 14.1. Biến thể: trả vị trí của cặp đầu tiên không khớp

Đề gốc chỉ yêu cầu boolean. Nhưng để giúp debug, ta có thể trả `(left,right)`
là chỉ số của hai ký tự alphanumeric đầu tiên khác nhau sau khi chuẩn hóa.
Nếu không có cặp sai thì trả `None`.
Điều đáng học: logic bỏ qua ký tự không đổi; thứ cần sửa chỉ là **thông tin return**.
Nếu input có dấu câu, các chỉ số trả về **nằm trên chuỗi gốc**,
không phải thứ tự vị trí của chuỗi đã lọc.
Ví dụ `'A!b:c'` đầu a/c sai ngay → `(0,4)`.

In [ ]:
def first_mismatch125(s):
    left,right=0,len(s)-1
    while left<right:
        if not s[left].isalnum():left+=1;continue
        if not s[right].isalnum():right-=1;continue
        if s[left].lower()!=s[right].lower():return (left,right)
        left+=1;right-=1
    return None
for sample in ('A!b:c','A!b:a','0P'):
    print(repr(sample),'vị trí sai:',first_mismatch125(sample))
assert first_mismatch125('A!b:c')==(0,4)

### 14.2. Biến thể: cho phép xóa TỐI ĐA một chữ/số

Nếu được xóa một ký tự alphanumeric, bài trở thành gần với **Valid Palindrome II**.
Khi một cặp khác nhau, ta chỉ có hai lựa chọn cần thử:
bỏ ký tự **bên trái** hoặc bỏ ký tự **bên phải**. Nếu một lựa chọn
làm đoạn còn lại đối xứng thì đạt. Nhưng với dấu câu, cần so theo chuỗi *đã lọc*;
đây là biến thể độc lập, **không phải yêu cầu** của #125.
Ta trình bày dưới dạng một bài luyện để chuyển giao kỹ thuật two pointers,
không lẫn logic mới vào hàm `class Solution` của đề gốc.

In [ ]:
def valid_after_one_deletion125(s):
    cleaned=''.join(c.lower() for c in s if c.isalnum())
    def is_pal_segment(l,r):
        while l<r:
            if cleaned[l]!=cleaned[r]:return False
            l+=1;r-=1
        return True
    l,r=0,len(cleaned)-1
    while l<r:
        if cleaned[l]!=cleaned[r]:
            return is_pal_segment(l+1,r) or is_pal_segment(l,r-1)
        l+=1;r-=1
    return True
for s0 in ('abca','racecar','abc','A!bca'):
    print(repr(s0),'<=1 lần xóa:',valid_after_one_deletion125(s0))

## 15. Hệ thống luyện tập — 6 bài từ cơ bản đến nâng cao

**Bài 1 — Dự đoán output Python:** `s = "0!P"`: hãy dự đoán `(s[0].isalnum(), s[1].isalnum(), s[2].lower())`.

**Bài 2 — Trace tay:** Lập bảng hai con trỏ của `"A!b:c"`: sau mỗi lượt ghi left/right trước, ký tự, hành động và left/right sau.

**Bài 3 — Hoàn thiện code chính:** tự viết hàm `exercise_palindrome125(s)` mà không nhìn cell nộp.

**Bài 4 — Biến thể:** Trả `(i,j)` là vị trí hai ký tự chữ/số đầu tiên không khớp ở chuỗi GỐC; trả `None` nếu không có cặp sai.

**Bài 5 — Viết test phản ví dụ:** Tự tạo ít nhất 5 `(s, expected)` bao gồm rỗng/toàn dấu câu, chuỗi chỉ có chữ số, ký tự hoa/thường, cặp sai và cặp đúng.

**Bài 6 — Mini challenge:** Cho phép xóa tối đa MỘT ký tự chữ/số sau khi chuẩn hóa, trả True/False. Gợi ý thử bỏ một trong hai ký tự ở cặp đầu tiên khác nhau.

**Cách tự học:** làm bài 1–2 trên giấy → viết TODO bài 3–5 → tự kiểm bằng test của chính mình → thử bài 6. Hãy ghi rõ lý do chọn trạng thái, ranh giới vòng lặp và điều kiện dừng; **đáp án tham khảo nằm sau toàn bộ đề bài**.

### Bài 3 — TODO: tự viết lại Valid Palindrome

Viết code vào hàm ở cell tiếp theo theo đúng kiểu input/output của đề.
Không gọi hàm TODO khi chạy toàn bộ notebook. Khi hoàn thành, tự bỏ `raise NotImplementedError` rồi chạy test được gợi ý dưới hàm.
**Tiêu chí tự chấm:** giải thích được vì sao code đúng ngay cả ở trường hợp nhỏ nhất, và không làm thao tác dư thừa so với lời giải chính.

In [ ]:
def exercise_palindrome125(s):
    # TODO 1: tạo hai chỉ số trái / phải
    # TODO 2: nếu ký tự một đầu không phải chữ/số, bỏ qua đầu đó
    # TODO 3: nếu cả hai hợp lệ, so sánh sau lower()
    # TODO 4: kết luận False nếu khác; nếu không còn cặp khác, True
    raise NotImplementedError("Tự hoàn thành trước khi chạy test")
# Khi tự làm xong, hãy chạy: assert exercise_palindrome125('0P') is False

### Bài 4 — TODO: đổi yêu cầu, giữ kỹ thuật cốt lõi

Trả `(i,j)` là vị trí hai ký tự chữ/số đầu tiên không khớp ở chuỗi GỐC; trả `None` nếu không có cặp sai.

Đừng chép y nguyên code submit: hãy chỉ ra đầu ra thay đổi ở đâu; tự tạo 2 input mới để kiểm thử.

In [ ]:
def exercise_first_mismatch125(s):
    # TODO: trả (left,right) tại cặp ALPHANUMERIC đầu tiên không khớp.
    # Nếu tất cả khớp, trả None.
    raise NotImplementedError("Chưa làm bài 4")
# TODO test sau khi hoàn thành: assert exercise_first_mismatch125('A!b:c') == (0,4)

### Bài 5 — TODO: tạo bộ kiểm thử có chủ đích

Tự tạo ít nhất 5 `(s, expected)` bao gồm rỗng/toàn dấu câu, chuỗi chỉ có chữ số, ký tự hoa/thường, cặp sai và cặp đúng.

Mỗi test phải có **expected tự tính trước**, không được lấy `expected=solve(input)`. Một test tốt thường có mục tiêu bắt một kiểu sai cụ thể. Nếu có node, dựng dữ liệu mới ở mỗi lượt test.

In [ ]:
def exercise_cases125():
    # TODO: trả list các tuple (chuỗi, bool_mong_đợi), ít nhất 5 trường hợp.
    # Không gọi solve() trong hàm này để tự tính expected trước.
    raise NotImplementedError("Chưa thiết kế test")
# Sau khi hoàn thành mới viết for s,e in exercise_cases125(): assert solve(s)==e

### Bài 6 — Mini challenge chuyển giao kỹ thuật (khó)

**Yêu cầu:** Cho phép xóa tối đa MỘT ký tự chữ/số sau khi chuẩn hóa, trả True/False. Gợi ý thử bỏ một trong hai ký tự ở cặp đầu tiên khác nhau.

    Bài 6 không thuộc đề gốc. Hãy tự phân tích **miền input, output, tính đúng**
    và chọn một phản ví dụ nhỏ trước khi code. Nếu thay đổi cấu trúc lời giải,
    cần nói rõ điều kiện nào đã thay đổi so với bài gốc.
    Không gọi hàm có TODO trong lần chạy toàn bộ notebook: để nguyên `NotImplementedError`
    cho tới khi bạn tự điền mã và tự bật test.
    **Kiểm tra:** so kết quả với đáp án cuối, nhưng chỉ sau khi đã tự làm.

In [ ]:
def exercise_one_deletion125(s):
    # TODO: chuẩn hóa string rồi dùng hai con trỏ.
    # Tại cặp khác nhau đầu tiên, thử bỏ một trong hai đầu.
    raise NotImplementedError("Mini challenge chưa giải")
# Sau khi giải: assert exercise_one_deletion125('abca') is True

## 16. Gợi ý theo ba cấp — chỉ đọc sau khi tự thử

**Cấp 1 — trực giác:** bỏ dấu câu/so không phân biệt hoa thường, rồi nghĩ đến cặp ngoài cùng.
**Cấp 2 — biến trạng thái:** `left=0`, `right=len(s)-1`, `while left<right`, `isalnum`, `lower`.
**Cấp 3 — gần code:** nếu một đầu không hợp lệ thì chỉ dịch đầu ấy và `continue`; chỉ so sau khi cả hai hợp lệ.
**Cấp nâng cao:** sau khi lọc, nếu gặp cặp khác nhau, bài xóa một ký tự chỉ có hai trường hợp cần thử: bỏ trái hoặc bỏ phải.

**Cách sử dụng gợi ý:** bắt đầu ở cấp 1; nếu vẫn chưa ra, chỉ mở cấp 2 rồi tự viết tiếp. Không đọc lời giải đầy đủ ngay khi vừa gặp khó. Với bài 6, thử tự xác định input nhỏ nhất gây sai trong phương án bạn đề xuất.

## 17. Đáp án tham khảo — chỉ đọc sau khi đã làm TODO

Các hàm **đáp án** dùng tên khác với `exercise_*` để không vô tình thực thi code chưa hoàn thiện.
Đáp án bài 3 được định nghĩa trước, rồi đến các biến thể bài 4–6 và bộ test tương ứng.
Sau khi chạy, hãy so sánh logic từng dòng với lời giải chính và tìm lý do các cài đặt tương đương, không chỉ so con số output.

In [ ]:
def exercise_core_answer(s):
    left,right=0,len(s)-1
    while left<right:
        if not s[left].isalnum():left+=1;continue
        if not s[right].isalnum():right-=1;continue
        if s[left].lower()!=s[right].lower():return False
        left+=1;right-=1
    return True

print("Đã định nghĩa đáp án tham khảo; xem lại từng dòng rồi tự gõ lại.")

### Đáp án bài 4 — biến thể với yêu cầu đầu ra mới

Bài 4 yêu cầu: Trả `(i,j)` là vị trí hai ký tự chữ/số đầu tiên không khớp ở chuỗi GỐC; trả `None` nếu không có cặp sai.

    So với lời giải gốc, hãy chỉ ra phần nào giữ nguyên:
    cú pháp duyệt, trạng thái trước/sau, bất biến. Phần thay đổi phải tương ứng
    đúng với **đầu ra mới**, không làm thay chữ ký code submit #
    hoặc vô tình lẫn thêm điều kiện ngoài phạm vi bài gốc.
    Đoạn đáp án dưới đây có assert riêng để kiểm tính đúng.

In [ ]:
def answer_first_mismatch125(s):
    left,right=0,len(s)-1
    while left<right:
        if not s[left].isalnum():left+=1;continue
        if not s[right].isalnum():right-=1;continue
        if s[left].lower()!=s[right].lower():return (left,right)
        left+=1;right-=1
    return None
for s,expected in [('A!b:c',(0,4)),('A!b:a',None),('0P',(0,1)),('',None)]:
    assert answer_first_mismatch125(s)==expected
print('PASS đáp án bài 4 — chỉ số cặp không khớp')

### Đáp án bài 6 và kiểm thử đáp án bài 5

**Bài 6:** Cho phép xóa tối đa MỘT ký tự chữ/số sau khi chuẩn hóa, trả True/False. Gợi ý thử bỏ một trong hai ký tự ở cặp đầu tiên khác nhau.

    Đọc suy luận trước khi so code: giữ trạng thái nào; tại mỗi bước
    trạng thái cập nhật ra sao; vì sao khi dừng, kết quả đáp ứng yêu cầu.
    Bài 5 là tự xây bộ test: đáp án tham khảo còn có các mẫu độc lập,
    chạy kèm assert nhưng **không hề chạy TODO của người học**.
    Nếu bộ test bạn tự nghĩ khác, điều đó bình thường: hãy đảm bảo cả
    test của bạn và test tham khảo đều có expected đúng.

In [ ]:
def answer_one_deletion125(s):
    clean=''.join(c.lower() for c in s if c.isalnum())
    def ok(l,r):
        while l<r:
            if clean[l]!=clean[r]:return False
            l+=1;r-=1
        return True
    l,r=0,len(clean)-1
    while l<r:
        if clean[l]!=clean[r]:return ok(l+1,r) or ok(l,r-1)
        l+=1;r-=1
    return True
for s,expected in [('abca',True),('abc',False),('aba',True),('!@',True),('A!bca',True)]:
    assert answer_one_deletion125(s)==expected
print('PASS đáp án mini challenge #125')

cases125=[('',True),('...',True),('0P',False),('Aa',True),('1b1',True),('abca',False)]
for phrase,expected in cases125:
    assert solve(phrase)==expected
print('PASS bộ test tham khảo bài 5')

### Lời giải thích và đáp án ngắn cho cả 6 bài

**Bài 1:** `(True, False, 'p')` vì số 0 được nhận, dấu `!` bị bỏ, chữ P đổi thành p.
**Bài 2:** tại `A/c` phải kết luận False ngay, không tiến thêm; những dấu câu bên trong không cần xét nữa.
**Bài 3:** tham khảo `exercise_core_answer` và cell submit ở trên; phải trả `bool` chứ không trả chuỗi đã chuẩn hóa.
**Bài 4:** `answer_first_mismatch125` trả chỉ số trên CHUỖI GỐC, khác chỉ số của chuỗi sau lọc.
**Bài 5:** bộ test ở cell cuối; kiểm tra dự đoán bằng tay rồi mới gọi `solve`.
**Bài 6:** `answer_one_deletion125`: khi gặp sai, hai khả năng xóa duy nhất là xóa một trong hai đầu. Không xóa 2 ký tự.

Đối với bài tập viết code, hãy so sánh cả **điều kiện lặp**, **thứ tự cập nhật** và **cách return** chứ không chỉ so giá trị cuối. Các hàm `answer_*` của bài 4 và 6 nằm ở những cell ngay phía trên để người học có thể chạy riêng sau khi tự làm.

## 18. Tóm tắt và checklist trước khi nộp

**Tư duy phải tự diễn đạt được bằng lời:** nêu bài toán bằng một câu, chỉ rõ trạng thái nào đủ để quyết định bước kế tiếp, rồi mới viết code. Đọc lại các ví dụ khó trong notebook; khi một điều kiện `if` làm bạn phân vân, hãy lập bảng giá trị **ngay trước và sau** câu lệnh ấy.

1. Tôi đã phân biệt đầu vào/đầu ra, và đọc đúng chữ ký `class Solution` của bài.
2. Tôi đã tự chạy thử ví dụ nhỏ, biên, phản ví dụ và kiểm thử đối chiếu nhiều cách giải.
3. Tôi hiểu từng cấu trúc Python, có thể giải thích vì sao dùng nó thay vì cách khác.
4. Tôi chỉ ra được bất biến: sau bước thứ k, điều gì **luôn đúng** và tại sao?
5. Tôi phân tích Time và Space dựa trên số lượng phần tử/nút/bít và có tính tới output/stack/chuỗi tạm.
6. Tôi đã hoàn thành TODO **trước** khi kéo tới đáp án tham khảo.
7. Tôi chỉ sao chép `class Solution` để nộp LeetCode; các hàm dựng input, trace và bộ test dành cho notebook, không dành cho submit.

**Tự đánh giá:** bạn hiểu bài khi tự viết được lời giải trên một trang trống và xử lý được một biến thể mà không sao chép mã cũ.

**Mẫu nhận dạng của bài 125:** chuỗi cần so đối xứng sau khi bỏ ký tự không liên quan → hai con trỏ hai đầu, time O(n), extra space O(1).